[![In Colab öffnen](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Y-Robin/DeepLearningKurs/blob/main/notebooks/Day_9/01_tag_9_objekterkennung_einfache_anker_nms.ipynb)

# Tag 9 – 01: Objekterkennung auf echten Bildern mit einfachen Ankern und NMS

Wir erkennen eine Person in realen Straßen- und Campus-Fotos aus Penn-Fudan.
Der **Detektionskopf** bleibt absichtlich klein und transparent: eine Klasse
(`Person`) und ein 4×4-Ankergitter. Damit die wenigen echten Trainingsbilder
ausreichen, verwendet er einen vortrainierten ResNet-18-Backbone und lernt
zunächst nur den neuen Detektionskopf.


## Lernziel: Was ist Input, Label und Vorhersage?

| Rolle | Form / Beispiel | Bedeutung |
| --- | --- | --- |
| Input `image` | `(3, 192, 192)` | echtes RGB-Foto ohne eingezeichnete Box |
| Label `box_xyxy` | `[x_min, y_min, x_max, y_max]` | aus der Originalmaske abgeleitete Personenbox |
| Trainingsziel | `(4, 4)` Objektkarte + `(4, 4, 4)` Offsets | eine positive Zelle und ihre Boxkorrektur |
| Modellvorhersage | `(4, 4, 5)` | pro Anker: Objekt-Logit plus `t_x, t_y, t_w, t_h` |

Ein **Anker** ist eine Startbox. Alle 16 Anker haben zunächst dieselbe Größe
(48×48 Pixel nach dem Resize) und liegen in den Zentren der 4×4-Zellen. Nur der
Anker in der Zelle mit dem Personenzentrum erhält das Label `1`. Das Modell
korrigiert dessen Mitte und Größe mit vier Offsets.

> Eine U-Net ist ein sehr gutes Modell für Segmentierung: Sie sagt eine Klasse
> für jedes Pixel vorher. Für dieses Lernziel – eine Rechteckbox mit einer
> Objektwahrscheinlichkeit – ist ein kleiner Box-Head direkter und leichter zu
> erklären.


In [ ]:
from pathlib import Path
import copy
import json
import random

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision.models import ResNet18_Weights, resnet18
from torchvision.transforms import functional as TF

RANDOM_STATE = 42
torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

IMAGE_SIZE = 192
GRID_SIZE = 4
ANCHOR_SIZE = 48 / IMAGE_SIZE  # normierte Breite und Höhe
BATCH_SIZE = 16
USE_PRETRAINED_BACKBONE = True
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_DIR = Path("datasets/PennFudanPed")

print("Device:", device)
print("PyTorch:", torch.__version__)


## Realen Datensatz laden und augmentieren

Starte zuerst `00_tag_9_datensatz_vorbereiten.ipynb`, falls die Annotationen
noch fehlen. Die Fotos werden nur für das Lernmodell auf 192×192 Pixel skaliert;
die Bounding Box wird dabei auf `[0, 1]` normiert.

Im Training gibt es zusätzlich zufälliges horizontales Spiegeln sowie leichte
Helligkeits- und Kontraständerungen. Beim Spiegeln wird die Box passend
umgerechnet. So entstehen plausible Varianten echter Fotos, ohne Labels zu
verfälschen.


In [ ]:
class PennFudanSinglePersonDataset(Dataset):
    def __init__(self, split, augment=False):
        annotation_path = DATA_DIR / f"{split}_annotations.json"
        if not annotation_path.exists():
            raise FileNotFoundError(
                f"{annotation_path} fehlt. Bitte zuerst das Notebook 00 ausführen."
            )
        self.annotations = json.loads(annotation_path.read_text(encoding="utf-8"))
        self.augment = augment

    def __len__(self):
        return len(self.annotations)

    def __getitem__(self, index):
        annotation = self.annotations[index]
        image = Image.open(DATA_DIR / annotation["image"]).convert("RGB")
        width, height = image.size
        image = TF.to_tensor(image)
        box = torch.tensor(annotation["box_xyxy"], dtype=torch.float32)
        box = box / torch.tensor([width, height, width, height], dtype=torch.float32)
        image = TF.resize(image, [IMAGE_SIZE, IMAGE_SIZE], antialias=True)

        if self.augment:
            if random.random() < 0.5:
                image = TF.hflip(image)
                x0, y0, x1, y1 = box
                box = torch.stack([1 - x1, y0, 1 - x0, y1])
            image = TF.adjust_brightness(image, brightness_factor=random.uniform(0.85, 1.15))
            image = TF.adjust_contrast(image, contrast_factor=random.uniform(0.85, 1.15))

        return image, box


train_dataset = PennFudanSinglePersonDataset("train", augment=True)
valid_dataset = PennFudanSinglePersonDataset("valid", augment=False)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

image, box = valid_dataset[0]  # ohne Augmentation: gut für eine lesbare Visualisierung
print("Input image nach Resize:", tuple(image.shape), image.dtype)
print("Label box_xyxy (normalisiert):", box.tolist())


In [ ]:
def anchors_for_grid(grid_size=GRID_SIZE, device="cpu"):
    # Mittelpunkte der gleich großen Startboxen, Form (G, G, 2).
    centers = (torch.arange(grid_size, device=device, dtype=torch.float32) + 0.5) / grid_size
    yy, xx = torch.meshgrid(centers, centers, indexing="ij")
    return torch.stack([xx, yy], dim=-1)


def xyxy_to_cxcywh(boxes):
    x0, y0, x1, y1 = boxes.unbind(dim=-1)
    return torch.stack([(x0 + x1) / 2, (y0 + y1) / 2, x1 - x0, y1 - y0], dim=-1)


def make_targets(boxes):
    # Ordnet jede echte Box genau einem Anker zu.
    batch_size = len(boxes)
    object_target = torch.zeros(batch_size, GRID_SIZE, GRID_SIZE, device=boxes.device)
    box_target = torch.zeros(batch_size, 4, GRID_SIZE, GRID_SIZE, device=boxes.device)
    centers = anchors_for_grid(device=boxes.device)

    cxcywh = xyxy_to_cxcywh(boxes)
    for batch_index, (cx, cy, width, height) in enumerate(cxcywh):
        col = torch.clamp((cx * GRID_SIZE).long(), 0, GRID_SIZE - 1)
        row = torch.clamp((cy * GRID_SIZE).long(), 0, GRID_SIZE - 1)
        anchor_cx, anchor_cy = centers[row, col]
        object_target[batch_index, row, col] = 1.0
        # Korrektur relativ zum Mittelpunkt bzw. zur festen Ankergröße.
        box_target[batch_index, :, row, col] = torch.stack([
            (cx - anchor_cx) * GRID_SIZE * 2,
            (cy - anchor_cy) * GRID_SIZE * 2,
            torch.log(width / ANCHOR_SIZE),
            torch.log(height / ANCHOR_SIZE),
        ])
    return object_target, box_target


def draw_box(ax, box_xyxy, color, label, linewidth=2):
    x0, y0, x1, y1 = (box_xyxy.detach().cpu().numpy() * IMAGE_SIZE)
    ax.add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0,
                               fill=False, edgecolor=color, linewidth=linewidth))
    ax.text(x0, max(2, y0 - 3), label, color=color, fontsize=9,
            bbox={"facecolor": "black", "alpha": 0.55, "pad": 1})


def show_input_and_label(image, box):
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.imshow(image.permute(1, 2, 0))
    for position in range(1, GRID_SIZE):
        ax.axhline(position * IMAGE_SIZE / GRID_SIZE, color="white", alpha=0.35)
        ax.axvline(position * IMAGE_SIZE / GRID_SIZE, color="white", alpha=0.35)
    draw_box(ax, box, "lime", "Label: echte Box")
    ax.set_title("Input (Bild) + Label (grün) + 16 gleich große Ankerzellen")
    ax.axis("off")
    plt.show()


show_input_and_label(image, box)
object_target, box_target = make_targets(box.unsqueeze(0))
print("Objekt-Labelkarte (1 = zuständiger Anker):\n", object_target[0].int())


## Transfer Learning: kleiner Detektionskopf auf vortrainiertem Backbone

Ein komplett neues CNN bekommt im Ein-Person-Split nur wenige hundert
Trainingsschritte – zu wenig, um auf realen Fotos robuste Bildmerkmale zu
lernen. Deshalb laden wir ResNet-18 mit ImageNet-Gewichten:

1. **Warm-up:** Der Backbone bleibt eingefroren; nur der neue 4×4-Box-Head lernt.
2. **Fine-Tuning:** Danach wird nur der letzte ResNet-Block mit kleiner Lernrate
   geöffnet.

`BatchNorm`, `Dropout2d` und `weight_decay` stabilisieren und regularisieren den
neuen Head. Die Ausgabe bleibt unverändert verständlich: fünf Zahlen pro Anker.
Beim ersten Durchlauf lädt PyTorch die ResNet-18-Gewichte einmalig in den Cache.


In [ ]:
class TransferAnchorDetector(nn.Module):
    def __init__(self, use_pretrained=True):
        super().__init__()
        weights = ResNet18_Weights.DEFAULT if use_pretrained else None
        resnet = resnet18(weights=weights)

        # ResNet ohne globale Mittelung und Klassifikationsschicht:
        # räumliche Features (Batch, 512, Höhe/32, Breite/32) bleiben erhalten.
        self.features = nn.Sequential(*list(resnet.children())[:-2])
        self.backbone_frozen = True
        for parameter in self.features.parameters():
            parameter.requires_grad = False

        # Aus den 512 ResNet-Features werden 5 Werte für jede der 4×4-Zellen.
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d((GRID_SIZE, GRID_SIZE)),  # immer genau 16 Ankerpositionen
            nn.Conv2d(512, 128, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Dropout2d(p=0.15),
            nn.Conv2d(128, 5, kernel_size=1),  # Objekt-Logit, tx, ty, tw, th
        )

        # Feste ImageNet-Normalisierung: wird gespeichert, aber nicht gelernt.
        self.register_buffer("image_mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("image_std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def forward(self, x):
        x = (x - self.image_mean) / self.image_std
        features = self.features(x)
        return self.head(features)  # (batch, 5, 4, 4)

    def unfreeze_last_block(self):
        # Nach dem Head-Warm-up nur ResNets letzten Block behutsam anpassen.
        self.backbone_frozen = False
        for parameter in self.features[-1].parameters():
            parameter.requires_grad = True

    def train(self, mode=True):
        super().train(mode)
        if self.backbone_frozen:
            self.features.eval()  # BatchNorm-Werte des eingefrorenen Backbones nicht verändern.
        else:
            self.features[:-1].eval()  # beim Fine-Tuning nur der letzte Block trainiert.
        return self


model = TransferAnchorDetector(use_pretrained=USE_PRETRAINED_BACKBONE).to(device)
print(model)
print("Trainierbare Parameter im Warm-up:",
      sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad))


### Die Klasse `TransferAnchorDetector` Schritt für Schritt

| Baustein | Ausgabe / Aufgabe | Warum er hier gebraucht wird |
| --- | --- | --- |
| `resnet18(weights=...)` | vortrainierte Bildmerkmale | Der Backbone hat bereits Kanten, Texturen und Objektteile aus vielen Bildern gelernt. |
| `children()[:-2]` | räumliche Feature Maps mit 512 Kanälen | Globale Mittelung und ImageNet-Klassifikator werden entfernt, damit Ortsinformation für Boxen erhalten bleibt. |
| eingefrorene `features` | zunächst keine Gradienten | Der kleine Datensatz überschreibt die nützlichen Vortrainingsmerkmale nicht sofort. |
| `AdaptiveAvgPool2d((4, 4))` | `(Batch, 512, 4, 4)` | Eine Zelle entspricht genau einer der 16 Ankerpositionen. |
| `Conv2d(512, 128)` + BatchNorm + ReLU | Detektionsmerkmale | Übersetzt allgemeine ResNet-Merkmale in Merkmale für Personenboxen. |
| `Dropout2d(0.15)` | zufällig fehlende Feature-Kanäle im Training | Regularisierung gegen Überanpassung auf die wenigen Fotos. |
| `Conv2d(128, 5, 1)` | `(Batch, 5, 4, 4)` | Je Anker: ein Objekt-Logit und vier Box-Offsets. |

### Warum genau diese ImageNet-Normalisierung?

Die Eingabepixel liegen zunächst in $[0,1]$. Für jeden RGB-Kanal berechnet das
Modell $x_{norm}=(x-\mu)/\sigma$ mit
$\mu=(0{,}485, 0{,}456, 0{,}406)$ und
$\sigma=(0{,}229, 0{,}224, 0{,}225)$. Diese Werte sind Mittelwert und
Standardabweichung der ImageNet-Trainingsbilder pro Rot-, Grün- und Blaukanal.
Der ResNet-18-Backbone wurde mit genau dieser Skala trainiert. Ohne dieselbe
Normalisierung wären seine ersten Aktivierungen anders skaliert und das
Vortraining deutlich weniger nützlich.

`register_buffer` speichert Mittelwert und Standardabweichung im Modellzustand und
verschiebt sie automatisch mit `model.to(device)`, aber sie sind keine trainierbaren
Parameter. `unfreeze_last_block()` öffnet erst nach dem Warm-up nur den letzten
ResNet-Block; mit kleiner Lernrate darf er sich dann an Personenfotos anpassen.
Die überschriebenen `train()`-Aufrufe halten die BatchNorm-Werte in den weiterhin
eingefrorenen ResNet-Blöcken stabil.


## Loss: Binärklassifikation + Box-Regression

Der Loss besteht aus zwei klar getrennten Teilen.

### 1. Objektwahrscheinlichkeit: Binary Cross Entropy (BCE)

Für jeden der 16 Anker sagt das Modell einen **Logit** $o_i$ voraus. Das Label
$y_i$ ist `1` für den Anker mit dem Personenzentrum und sonst `0`. Erst innerhalb
von `BCEWithLogitsLoss` wird daraus $p_i=\sigma(o_i)$.

$$L_{BCE} = -\frac{1}{16B}\sum_i w_i\left[y_i\log(p_i) + (1-y_i)\log(1-p_i)\right]$$

Positiver Anker: $w_i=4$. Negativer Anker: $w_i=1$. Das gleicht aus, dass pro
Bild nur ein positiver, aber 15 negative Anker vorkommen.

Dabei ist $B$ die **Batchgröße**, also die Anzahl der Bilder, die gleichzeitig
durch das Modell laufen. Bei `BATCH_SIZE = 16` ist $B=16$ (im letzten Batch kann
es weniger sein).

### 2. Bounding Box: Root Mean Squared Error (RMSE)

Der positive Anker sagt vier Offsets $(t_x,t_y,t_w,t_h)$ voraus. Wir vergleichen
nur dort Vorhersage $\hat{t}$ und Label $t$:

$$L_{RMSE}=\sqrt{\frac{1}{4N_{pos}}\sum_{i:y_i=1}\sum_{k=1}^{4}(\hat{t}_{i,k}-t_{i,k})^2 + \varepsilon}$$

$N_{pos}$ ist die Zahl der **positiven Anker im Batch**. Da unser vorbereiteter
Split genau eine Person pro Bild enthält, gilt normalerweise $N_{pos}=B$. Der
Faktor 4 im Nenner steht für die vier Boxwerte $t_x,t_y,t_w,t_h$.

RMSE ist die Quadratwurzel des mittleren quadratischen Fehlers. Ein Fehler von
beispielsweise `0,10` in den normierten Offsets bleibt dadurch als ungefähr
`0,10` interpretierbar. In großen Produktivmodellen ist `SmoothL1`/Huber oft
robuster gegen Ausreißer; hier verwenden wir RMSE, weil der Zusammenhang zwischen
Boxfehler und Loss sehr direkt sichtbar ist.

### 3. Gemeinsames Optimierungsziel

$$L_{gesamt}=L_{BCE}+8\cdot L_{RMSE}$$

Der Faktor 8 sorgt dafür, dass die vier Koordinaten nicht von den vielen
Klassifikationsentscheidungen überdeckt werden.


In [ ]:
object_loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([4.0], device=device))
BOX_LOSS_WEIGHT = 8.0


def masked_rmse(predicted_offsets, target_offsets, positive_mask):
    # Nur positive Anker und deren vier Koordinaten gehen in die Box-Regression ein.
    squared_error = ((predicted_offsets - target_offsets) ** 2) * positive_mask
    number_of_coordinates = positive_mask.sum() * predicted_offsets.shape[1]
    mean_squared_error = squared_error.sum() / number_of_coordinates.clamp_min(1)
    return torch.sqrt(mean_squared_error + 1e-8)


def detection_loss(raw_output, boxes):
    object_target, box_target = make_targets(boxes)
    object_logits = raw_output[:, 0]
    predicted_offsets = raw_output[:, 1:]
    loss_bce = object_loss_fn(object_logits, object_target)

    positive_mask = object_target.unsqueeze(1)  # (B, 1, G, G)
    loss_rmse = masked_rmse(predicted_offsets, box_target, positive_mask)
    total_loss = loss_bce + BOX_LOSS_WEIGHT * loss_rmse
    return total_loss, loss_bce.detach(), loss_rmse.detach()


def train_one_epoch(model, loader, optimizer):
    model.train()
    totals = torch.zeros(3)
    for images, boxes in loader:
        images, boxes = images.to(device), boxes.to(device)
        optimizer.zero_grad()
        raw_output = model(images)
        loss, loss_bce, loss_rmse = detection_loss(raw_output, boxes)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        totals += torch.tensor([loss.item(), loss_bce.item(), loss_rmse.item()]) * len(images)
    return (totals / len(loader.dataset)).tolist()


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    totals = torch.zeros(3)
    for images, boxes in loader:
        raw_output = model(images.to(device))
        loss, loss_bce, loss_rmse = detection_loss(raw_output, boxes.to(device))
        totals += torch.tensor([loss.item(), loss_bce.item(), loss_rmse.item()]) * len(images)
    return (totals / len(loader.dataset)).tolist()


In [ ]:
EPOCHS = 45
WARMUP_EPOCHS = 10
EARLY_STOPPING_PATIENCE = 10

def make_optimizer(fine_tuning=False):
    if not fine_tuning:
        return torch.optim.AdamW(model.head.parameters(), lr=1e-3, weight_decay=1e-4)
    return torch.optim.AdamW([
        {"params": model.head.parameters(), "lr": 5e-4},
        {"params": model.features[-1].parameters(), "lr": 1e-4},
    ], weight_decay=1e-4)


optimizer = make_optimizer(fine_tuning=False)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3)
history = {"train": [], "valid": []}
best_loss = float("inf")
best_state = None
epochs_without_improvement = 0

for epoch in range(1, EPOCHS + 1):
    if epoch == WARMUP_EPOCHS + 1:
        model.unfreeze_last_block()
        optimizer = make_optimizer(fine_tuning=True)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3)
        print("\nFine-Tuning: letzter ResNet-Block wird mit kleiner Lernrate trainiert.")

    train_values = train_one_epoch(model, train_loader, optimizer)
    valid_values = evaluate(model, valid_loader)
    scheduler.step(valid_values[0])
    history["train"].append(train_values)
    history["valid"].append(valid_values)
    learning_rate = optimizer.param_groups[0]["lr"]
    print(f"Epoche {epoch:02d}: train L={train_values[0]:.3f}, valid L={valid_values[0]:.3f} "
          f"(BCE={valid_values[1]:.3f}, RMSE={valid_values[2]:.3f}, lr={learning_rate:.1e})")

    if valid_values[0] < best_loss - 1e-4:
        best_loss = valid_values[0]
        best_state = copy.deepcopy(model.state_dict())
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
            print(f"Early Stopping nach {epoch} Epochen; beste Validierungs-Loss: {best_loss:.3f}")
            break

model.load_state_dict(best_state)
print(f"Bestes Modell wiederhergestellt (Validierungs-Loss: {best_loss:.3f}).")

plt.figure(figsize=(8, 4))
plt.plot([row[0] for row in history["train"]], marker="o", label="Training: Gesamt-Loss")
plt.plot([row[0] for row in history["valid"]], marker="o", label="Validierung: Gesamt-Loss")
plt.axvline(WARMUP_EPOCHS - 1, color="gray", linestyle="--", label="Start Fine-Tuning")
plt.xlabel("Epoche")
plt.ylabel("Loss")
plt.title("Transfer Learning mit Augmentation, Regularisierung und Early Stopping")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


## Aus fünf Rohwerten werden Kandidatenboxen

Zuerst wird für jeden der 16 Anker die Wahrscheinlichkeit `sigmoid(logit)`
berechnet. Danach werden die vier Offsets in eine Box zurückübersetzt. Die
Tabelle darunter zeigt jede Kandidatenbox einzeln, absteigend nach
Objektwahrscheinlichkeit.


In [ ]:
def decode_predictions(raw_output):
    # Dekodiert (B, 5, G, G) nach Boxen (B, G*G, 4) und Scores (B, G*G).
    batch_size = raw_output.shape[0]
    centers = anchors_for_grid(device=raw_output.device)
    object_scores = torch.sigmoid(raw_output[:, 0])
    offsets = raw_output[:, 1:].permute(0, 2, 3, 1)  # (B, G, G, 4)

    cx = centers[..., 0] + torch.tanh(offsets[..., 0]) / (2 * GRID_SIZE)
    cy = centers[..., 1] + torch.tanh(offsets[..., 1]) / (2 * GRID_SIZE)
    width = ANCHOR_SIZE * torch.exp(offsets[..., 2].clamp(-1.5, 1.5))
    height = ANCHOR_SIZE * torch.exp(offsets[..., 3].clamp(-1.5, 1.5))
    boxes = torch.stack([cx - width / 2, cy - height / 2,
                         cx + width / 2, cy + height / 2], dim=-1).clamp(0, 1)
    return boxes.reshape(batch_size, -1, 4), object_scores.reshape(batch_size, -1)


@torch.no_grad()
def predict_candidates(model, images):
    model.eval()
    return decode_predictions(model(images.to(device)))


images, true_boxes = next(iter(valid_loader))
candidate_boxes, candidate_scores = predict_candidates(model, images[:1])
order = candidate_scores[0].argsort(descending=True)

print("Anker | Wahrscheinlichkeit | vorhergesagte Box [x0, y0, x1, y1] (Pixel)")
for anchor_index in order.tolist():
    score = candidate_scores[0, anchor_index].item()
    pixel_box = (candidate_boxes[0, anchor_index] * IMAGE_SIZE).round().int().tolist()
    print(f"{anchor_index:>5} | {score:>16.3f} | {pixel_box}")


# Die höchste Wahrscheinlichkeit wählt eine Rasterzelle, auch wenn sie noch
# unter dem späteren 50-%-Cutoff liegt. So wird die Entscheidung sichtbar.
selected_anchor = int(order[0])
selected_row, selected_col = divmod(selected_anchor, GRID_SIZE)
anchor_center = torch.tensor([
    (selected_col + 0.5) / GRID_SIZE,
    (selected_row + 0.5) / GRID_SIZE,
])
predicted_box = candidate_boxes[0, selected_anchor].cpu()
predicted_center = xyxy_to_cxcywh(predicted_box.unsqueeze(0))[0, :2]
true_center = xyxy_to_cxcywh(true_boxes[0].unsqueeze(0))[0, :2]

fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(images[0].permute(1, 2, 0))
cell_width = IMAGE_SIZE / GRID_SIZE
ax.add_patch(plt.Rectangle((selected_col * cell_width, selected_row * cell_width),
                           cell_width, cell_width, facecolor="deepskyblue", alpha=0.20,
                           edgecolor="deepskyblue", linewidth=3))
for position in range(1, GRID_SIZE):
    ax.axhline(position * cell_width, color="white", alpha=0.45)
    ax.axvline(position * cell_width, color="white", alpha=0.45)
draw_box(ax, true_boxes[0], "lime", "Label-Box")
draw_box(ax, predicted_box, "deepskyblue", f"Anker {selected_anchor}: {candidate_scores[0, selected_anchor]:.2f}")
ax.scatter(*(anchor_center * IMAGE_SIZE), marker="x", s=110, color="deepskyblue", linewidths=3,
           label="Mitte der gewählten Rasterzelle")
ax.scatter(*(predicted_center * IMAGE_SIZE), marker="o", s=55, color="deepskyblue",
           label="vom Netz korrigierter Mittelpunkt")
ax.scatter(*(true_center * IMAGE_SIZE), marker="o", s=55, color="lime",
           label="Mittelpunkt der Label-Box")
ax.set_title(f"Gewählte Zelle: Zeile {selected_row}, Spalte {selected_col}")
ax.legend(loc="lower right", fontsize=8)
ax.axis("off")
plt.show()


In [ ]:
SCORE_THRESHOLD = 0.50
NMS_IOU_THRESHOLD = 0.30
MAX_DETECTIONS = 1  # Der vorbereitete Split enthält genau eine Person pro Bild.


def box_iou(one_box, many_boxes):
    # IoU einer Box mit vielen Boxen, alle im xyxy-Format.
    top_left = torch.maximum(one_box[:2], many_boxes[:, :2])
    bottom_right = torch.minimum(one_box[2:], many_boxes[:, 2:])
    intersection = (bottom_right - top_left).clamp(min=0).prod(dim=1)
    area_one = (one_box[2:] - one_box[:2]).clamp(min=0).prod()
    area_many = (many_boxes[:, 2:] - many_boxes[:, :2]).clamp(min=0).prod(dim=1)
    return intersection / (area_one + area_many - intersection + 1e-6)


def simple_nms(boxes, scores, score_threshold=SCORE_THRESHOLD,
               iou_threshold=NMS_IOU_THRESHOLD, max_detections=MAX_DETECTIONS):
    # Behält hochwahrscheinliche, nicht überlappende Boxen; hier höchstens eine.
    remaining = torch.where(scores >= score_threshold)[0]
    remaining = remaining[scores[remaining].argsort(descending=True)]
    keep = []
    while len(remaining) > 0 and len(keep) < max_detections:
        current = remaining[0]
        keep.append(current)
        if len(remaining) == 1:
            break
        overlaps = box_iou(boxes[current], boxes[remaining[1:]])
        remaining = remaining[1:][overlaps < iou_threshold]
    return torch.stack(keep) if keep else torch.empty(0, dtype=torch.long, device=boxes.device)


def visualise_selection(image, true_box, boxes, scores):
    keep = simple_nms(boxes, scores)
    fig, axes = plt.subplots(1, 2, figsize=(11, 5))
    for ax in axes:
        ax.imshow(image.permute(1, 2, 0))
        ax.axis("off")

    # Links: Kandidaten ab 50 % zeigen. Dicke kodiert die Objektwahrscheinlichkeit.
    for index in torch.where(scores >= SCORE_THRESHOLD)[0].tolist():
        score = scores[index].item()
        draw_box(axes[0], boxes[index], "orange", f"{index}: {score:.2f}",
                 linewidth=1 + 2 * score)
    draw_box(axes[0], true_box, "lime", "echtes Label", linewidth=2)
    axes[0].set_title("Kandidaten ab 50 % vor NMS (orange) und Label (grün)")

    for index in keep.tolist():
        score = scores[index].item()
        draw_box(axes[1], boxes[index], "deepskyblue", f"behalten: {score:.2f}", linewidth=3)
    draw_box(axes[1], true_box, "lime", "echtes Label", linewidth=2)
    axes[1].set_title("Nach NMS: genau eine stärkste Box")
    plt.tight_layout()
    plt.show()
    return keep


kept_indices = visualise_selection(images[0], true_boxes[0],
                                   candidate_boxes[0].cpu(), candidate_scores[0].cpu())
print("Von NMS behaltene Anker:", kept_indices.tolist())


## Die Auswahl der wichtigsten Box: 50-%-Cutoff und NMS

Für dieses Notebook ist ein Cutoff von **0,50** passend: Kandidaten mit weniger
als 50 % Objektwahrscheinlichkeit werden nicht gezeichnet und nicht an NMS
übergeben. Ein höherer Cutoff reduziert Fehlalarme, kann aber echte Personen
verwerfen, wenn ein Modell noch unsicher ist. In einem echten Mehrpersonen-Setup
würde man diesen Wert auf einem Validierungsset abstimmen.

Danach arbeitet **Non-Maximum Suppression (NMS)** nach der Objektwahrscheinlichkeit:

1. Kandidaten unter 0,50 fallen weg.
2. Die höchste verbleibende Box wird behalten.
3. Stark überlappende Boxen (IoU ≥ 0,30) werden unterdrückt.
4. Weil dieser Split genau eine Person pro Bild enthält, wird spätestens nach
   einer Box beendet.

Die **Intersection over Union** (IoU) ist
$$\operatorname{IoU}(A,B)=\frac{\text{Fläche}(A\cap B)}{\text{Fläche}(A\cup B)}.$$


In [ ]:
# Vier unabhängige Validierungsbilder: nur die beste NMS-Box gegen das Label.
images, true_boxes = next(iter(valid_loader))
candidate_boxes, candidate_scores = predict_candidates(model, images[:4])
fig, axes = plt.subplots(2, 2, figsize=(9, 9))

for ax, image, true_box, boxes, scores in zip(
    axes.flat, images[:4], true_boxes[:4], candidate_boxes.cpu(), candidate_scores.cpu()
):
    kept = simple_nms(boxes, scores)
    ax.imshow(image.permute(1, 2, 0))
    draw_box(ax, true_box, "lime", "Label")
    if len(kept):
        best = kept[0]
        draw_box(ax, boxes[best], "deepskyblue", f"Vorhersage {scores[best]:.2f}")
    ax.axis("off")
plt.suptitle("Grün = Label, Blau = nach NMS gewählte Vorhersage", y=0.92)
plt.tight_layout()


## Optional: ein bereits trainiertes PyTorch-Modell

Das Notebook oben trainiert absichtlich den kleinen, transparenten Detektor.
Die folgende Zelle lädt zusätzlich ein auf COCO vortrainiertes
`FasterRCNN-MobileNetV3` aus `torchvision` und wendet es auf das eingebaute
Beispielbild `astronaut` an. Beim ersten Ausführen lädt PyTorch Gewichte aus
dem Internet. Dieses reale Modell nutzt **mehrere Skalen und viele Anker** –
genau die Komplexität, die wir im Hauptbeispiel zunächst weglassen.

Falls der Gewichte-Download nicht möglich ist, beendet die Zelle den optionalen
Vergleich mit einer verständlichen Meldung; alle vorherigen Teile funktionieren
vollständig offline nach Notebook 00.


In [ ]:
RUN_PRETRAINED_DEMO = True

if RUN_PRETRAINED_DEMO:
    try:
        from skimage import data
        from torchvision.models.detection import (
            FasterRCNN_MobileNet_V3_Large_320_FPN_Weights,
            fasterrcnn_mobilenet_v3_large_320_fpn,
        )

        weights = FasterRCNN_MobileNet_V3_Large_320_FPN_Weights.DEFAULT
        pretrained_model = fasterrcnn_mobilenet_v3_large_320_fpn(weights=weights).to(device).eval()
        natural_image = TF.to_tensor(data.astronaut())
        with torch.no_grad():
            output = pretrained_model([natural_image.to(device)])[0]

        categories = weights.meta["categories"]
        selected = torch.where(output["scores"].cpu() >= 0.70)[0]
        fig, ax = plt.subplots(figsize=(6, 6))
        ax.imshow(natural_image.permute(1, 2, 0))
        for index in selected.tolist():
            x0, y0, x1, y1 = output["boxes"][index].cpu().tolist()
            name = categories[output["labels"][index].item()]
            score = output["scores"][index].item()
            ax.add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0,
                                       fill=False, edgecolor="deepskyblue", linewidth=2))
            ax.text(x0, max(2, y0 - 3), f"{name}: {score:.2f}", color="deepskyblue",
                    bbox={"facecolor": "black", "alpha": 0.55, "pad": 1})
        ax.set_title("Vortrainiertes Torchvision-Modell: Boxen nach internem NMS")
        ax.axis("off")
        plt.show()
    except Exception as error:
        print("Optionales vortrainiertes Modell konnte nicht geladen werden:")
        print(type(error).__name__ + ":", error)


## Zusammenfassung

- Das Bild ist der **Input**; die echte Bounding Box ist das **Label**.
- Jeder feste Anker sagt separat Objektwahrscheinlichkeit und vier
  Boxkorrekturen voraus.
- Die Loss verbindet Klassifikation aller Anker mit Boxregression nur am
  positiven Anker.
- NMS macht aus überlappenden Kandidaten die wichtigsten Boxen.

Als nächste Erweiterung kann der Datensatz zwei Objekte pro Bild enthalten.
Dann würden mehrere Zellen positive Labels bekommen und NMS könnte mehrere
räumlich getrennte Boxen behalten.
